In [7]:
from openai.types import model
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp import ClientSession

load_dotenv()




True

In [27]:
def calculator(expression: str):
    ''' use this tool to calculator tool for mathametical expressions '''
    try:
        return eval(expression, {"__builtins__": {}})
    except Exception as e:
        return f"Error: {e}"



In [28]:
def mock_llm(messages):
    last_user = messages[-1]["content"].lower()

    if "2+2" in last_user or "calculate" in last_user:
        return {
            "type": "tool_call",
            "tool": "calculator",
            "args": {"expression": "2+2"}
        }

    return {
        "type": "final",
        "content": "I don't need a tool."
    }


In [29]:
messages = [
    {"role": "user", "content": "calculate 2+2"}
]

while True:
    response = mock_llm(messages)
    print("MODEL:", response)

    if response["type"] == "final":
        print("FINAL:", response["content"])
        break

    if response["type"] == "tool_call":
        tool_name = response["tool"]
        args = response["args"]

        if tool_name == "calculator":
            result = calculator(**args)
        else:
            result = f"Unknown tool: {tool_name}"

        print("TOOL RESULT:", result)

        messages.append({
            "role": "tool",
            "name": tool_name,
            "content": str(result)
        })


MODEL: {'type': 'tool_call', 'tool': 'calculator', 'args': {'expression': '2+2'}}
TOOL RESULT: 4
MODEL: {'type': 'final', 'content': "I don't need a tool."}
FINAL: I don't need a tool.


In [ ]:
# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "calculator",
#             "description": "Evaluate a simple math expression.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {
#                     "expression": {
#                         "type": "string",
#                         "description": "A math expression like 2+2"
#                     }
#                 },
#                 "required": ["expression"]
#             }
#         }
#     }
# ]


In [31]:
messages = [
    {"role": "user", "content": "What is 17 * 24?"}
]

In [8]:
# 1. Define server parameters to run server.py using the workspace python environment
server_params = StdioServerParameters(
    command=".venv/bin/python",
    args=["server.py"]
)
# 2. Connect via stdio_client
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # 3. Initialize the session (Required before running queries)
        await session.initialize()
        
        # 4. Retrieve the tools from the server
        tools_result = await session.list_tools()
        
        # 5. Format MCP tools to the OpenAI/Nemotron format expected by the next cell
        tools = [
            {
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description,
                    "parameters": tool.inputSchema  # Map inputSchema to parameters
                }
            }
            for tool in tools_result.tools
        ]
print(tools)

[{'type': 'function', 'function': {'name': 'calculator', 'description': 'Evaluate a simple mathematical expression.', 'parameters': {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculatorArguments', 'type': 'object'}}}]


In [32]:
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nemotron_api_key")
)
gpt_client='gpt-4o-mini'

while True:
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=messages,
        tools=tools,
        temperature=0.5,
        max_tokens=1024
    )

    message = response.choices[0].message

    if message.tool_calls:
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            if tool_name == "calculator":
                result = calculator(**args)
            else:
                result = f"Unknown tool: {tool_name}"

            messages.append({
                "role": "assistant",
                "tool_calls": [tool_call]
            })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })
    else:
        print(message.content)
        break
print(response.choices[0].message.content)

17 × 24 = **408**
17 × 24 = **408**
